# 02 Coding Examples

This notebook covers three workflows:
1. Generative coding with LLM prompts
2. Structured parsing with Pydantic
3. Retrieval-Augmented Generation (RAG) from a PDF

⚠ Don't forget to select the CESAB 2026 virtual environment kernel on the upper-right hand side.

If you want to use local Ollama models, install Ollama first and pull the model weights (detailed in the `INSTALL_PYTHON` file)

In [1]:
import sys
from pathlib import Path
from typing import Annotated, List, Literal, Union

from pydantic import BaseModel, Field
from transformers import pipeline

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFacePipeline

print(sys.version)
print(sys.executable)


c:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
c:\Users\Devi\R-working-folder\CESAB_AI_lit_review_spring_2026\.venv\Scripts\python.exe


In [ ]:
# # Fallback model (works without Ollama, but may be slower than API-based models).
# gen_pipe = pipeline("text2text-generation", model="google/flan-t5-base", device=-1)
# model1 = HuggingFacePipeline(pipeline=gen_pipe)
# model2 = HuggingFacePipeline(pipeline=gen_pipe)

Ollama cell below. Use this if you have Ollama running locally.

In [2]:
from langchain_ollama.llms import OllamaLLM
model1 = OllamaLLM(model="deepseek-r1:1.5b", temperature=0.2)
model2 = OllamaLLM(model="phi3:3.8b", temperature=0.2)

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
     You are a scientist performing a literature review.
     Task: Identify environmental impacts studied in the article text.
     Return a summary of impacts in 10 words or less.
     Do not invent details.
     """
    ),
    ("human", "Article text: {context}")
])

input_text = "This study researches impacts of pile-driving noise on marine mammals."
chain = prompt | model1
result = chain.invoke({"context": input_text})
print(result)

Pile-driving noise reduces marine mammal biodiversity by disrupting their feeding patterns and habitat availability.


In [5]:
class Location(BaseModel):
    location: str = Field(default="None", description="Study location")
    year: int = Field(default="None", description="Study year in yyyy format")

location_parser = PydanticOutputParser(pydantic_object=Location)

prompt_location = ChatPromptTemplate.from_messages([
    ("system",
     """
     Identify location and year from article text.
     Do not invent details.
     Wrap output in json tags: {format_instructions}
     """
    ),
    ("human", "Article text: {context}")
]).partial(format_instructions=location_parser.get_format_instructions())

raw_result = (prompt_location | model1).invoke({"context": "The study was conducted in Hawaii in 2015."})
raw_text = raw_result.content if hasattr(raw_result, "content") else raw_result

# Parse directly into the Pydantic schema.
structured_result = location_parser.parse(raw_text)
print(structured_result.model_dump())


{'location': 'Hawaii', 'year': 2015}


In [10]:
# Multi-label structured parsing example with LLM repair pass
from typing import TypeVar

T = TypeVar("T", bound=BaseModel)


class PressureType(BaseModel):
    pressure_type: List[Literal["Ships", "Marine_renewable_energy"]] = Field(default_factory=list)


pressure_parser = PydanticOutputParser(pydantic_object=PressureType)

prompt_pressure = ChatPromptTemplate.from_messages([
    ("system",
     """
     You are a scientist performing a scientific literature review.

     Task:
     Return ALL pressure types present in the article text.

     Allowed labels ONLY:
     - Ships
     - Marine_renewable_energy

     Rules:
     - Return only labels supported by the text.
     - If both are present, return both.
     - If none are present, return an empty list.
     - Return only valid JSON that follows: {format_instructions}
     """
    ),
    ("human", "Article text: {context}")
]).partial(format_instructions=pressure_parser.get_format_instructions())


def parse_with_llm_fix(
    parser: PydanticOutputParser,
    raw_output,
    fixer_llm,
):
    """Try strict parser first; if it fails, ask a second LLM to repair into valid parser JSON."""
    raw_text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)

    try:
        return parser.parse(raw_text)
    except Exception as first_error:
        fix_prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """
                You repair model outputs so they exactly match a required JSON schema.
                Return only valid JSON. No markdown, no explanation.
                Required schema instructions:
                {format_instructions}
                """,
            ),
            (
                "human",
                """
                Original model output:
                {bad_output}

                Parser error:
                {error_message}

                Rewrite the output so it is valid for the schema.
                """,
            ),
        ]).partial(format_instructions=parser.get_format_instructions())

        fixed_output = (fix_prompt | fixer_llm).invoke(
            {
                "bad_output": raw_text,
                "error_message": str(first_error),
            }
        )
        fixed_text = fixed_output.content if hasattr(fixed_output, "content") else str(fixed_output)

        return parser.parse(fixed_text)


text_pressure = "This study investigates environmental impacts from shipping and marine renewable energy."
raw_pressure = (prompt_pressure | model1).invoke({"context": text_pressure})
print(f"raw result:\n{raw_pressure}\n")

# Use model2 (e.g., phi3) as a repair model when first parse fails.
parsed_pressure = parse_with_llm_fix(pressure_parser, raw_pressure, fixer_llm=model2)
print(f"formatted result:\n{parsed_pressure.model_dump()}")


raw result:
The article text discusses both ships and marine renewable energy. Both terms are valid pressure types as per the allowed labels.

```json
{
  "pressure_type": ["Ships", "Marine_renewable_energy"]
}
```

formatted result:
{'pressure_type': ['Ships', 'Marine_renewable_energy']}


## RAG Example


Retrieval-Augmented Generation (RAG) allows an LLM to reference a specific knowledge base (context) outside of its training data sources before generating a response. This allows you to ask an LLM to answer questions about a source text.

This section loads a PDF from `data/raw-data/test_pdf.pdf`, chunks it, and uses FAISS retrieval from the embeddings model.

How to pick the best embedding model? You can see which ones are performing well on the [Hugging Face Leaderboard](https://huggingface.co/spaces/mteb/leaderboard)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from pypdf import PdfReader

# define an embeddings model
embeddings = OllamaEmbeddings(model="deepseek-r1:1.5b")

# Load the pdf text
pdf_path = Path("../data/raw-data/test_pdf.pdf")
reader = PdfReader(str(pdf_path))
text = reader.pages[0].extract_text()

# Split the text into chunks

# Define a function to split the text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " "],
)
# Use the function to chunk the text
chunks = splitter.create_documents([text])
chunk_text = [doc.page_content for doc in chunks]
len(chunk_text)


8

In [ ]:
# Define where to store your embeddings
faiss_dir = Path("../data/derived-data/test_pdf_faiss_vecs")

# If you've already extracted the embeddings, load them
# If not, get them using the embeddings model and save
if faiss_dir.exists():
    vectorstore = FAISS.load_local(str(faiss_dir), embeddings, allow_dangerous_deserialization=True)
else:
    vectorstore = FAISS.from_texts(texts=chunk_text, embedding=embeddings)
    vectorstore.save_local(str(faiss_dir))

# Using the embeddings model, find the chunks most similar to your query
# these will be used as the 'context' for your LLM
question = "Summarise in 10-100 words what this study is about."
related_docs = vectorstore.similarity_search(question, k=5)
context = "\n\n".join(doc.page_content for doc in related_docs)
print(context[:1200])

increased density and competitive position of oak reproduction. # 2002 Elsevier Science B.V. All rights reserved.

Forest Ecology and Management 163 (2002) 205–215




  Composition, diversity, and height of tree regeneration, 3 years
      after soil scarification in a mixed-oak shelterwood
                                                      James J. Zaczek*
                         Department of Forestry, Southern Illinois University, Carbondale, IL 62901-4411, USA
                                          Received 10 February 2001; accepted 23 April 2001



Abstract

0378-1127/02/$ – see front matter # 2002 Elsevier Science B.V. All rights reserved.
PII: S 0 3 7 8 - 1 1 2 7 ( 0 1 ) 0 0 5 8 0 - 1

1. Introduction                                                        succession to shade tolerant species prevails in many
                                                                       oak stands (Crow, 1988; Widmann, 1995; Abrams,
   Presently, northern red oak (Quercus rubra 

In [ ]:
## Now feed the text chunks to the LLM along with a question to answer

prompt_rag = ChatPromptTemplate.from_messages([
    ("system",
     """
     You are a scientific research assistant.
     Answer the question from the provided article text only.
     Do not invent details.
     """
    ),
    ("human", "Article text: {context}\nQuestion: {question}")
])

rag_chain = prompt_rag | model1
rag_result = rag_chain.invoke({"context": context, "question": question})
print(rag_result)

The study investigates the regeneration of northern red oak (Quercus rubra L.) after soil scarification, focusing on its reproduction and survival towards shade-tolerant species. It highlights that despite being in an xeric area, these oaks successfully regrow and reproduce, maintaining a competitive position in the forest ecosystem.


## Pretrained classifier example

You can also run existing models from Hugging Face without fine-tuning.

Explore models available on [HuggingFace](https://huggingface.co/models) 

In [ ]:
# using a pretrained model for text classification - sentiment analysis
# Try playing around with different wordings to see if you get a different result
# e.g. "Conserving blue carbon ecosystems is a great solution to climate change."
text_example = "Conserving blue carbon ecosystems can prevent climate emissions."


## Get model weights

# Topic model
topic_path = "cardiffnlp/tweet-topic-latest-multi"
topic_task = pipeline(task="text-classification", model=topic_path, tokenizer=topic_path)

# Sentiment model
sentiment_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_task = pipeline("sentiment-analysis", model=sentiment_path, tokenizer=sentiment_path)


## Get topic model predictions
# `top_k=None` requests scores for all labels in recent Transformers versions.
raw_topic = topic_task(text_example, top_k=None)

# Normalize topic output shape across Transformers versions.
if isinstance(raw_topic, dict):
    topic_items = [raw_topic]
elif isinstance(raw_topic, list) and raw_topic and isinstance(raw_topic[0], list):
    topic_items = raw_topic[0]
elif isinstance(raw_topic, list):
    topic_items = raw_topic
else:
    topic_items = []

labels = {
    item["label"]
    for item in topic_items
    if isinstance(item, dict) and item.get("score", 0) > 0.5
}

## Get sentiment model predictions
raw_sentiment = sentiment_task(text_example)
sentiment_result = raw_sentiment[0] if isinstance(raw_sentiment, list) else raw_sentiment

## Print outputs from both classification models
print(labels)
print(sentiment_result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/tweet-topic-latest-multi
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'science_&_technology', 'news_&_social_concern'}
{'label': 'neutral', 'score': 0.7538139820098877}
